# Saving and Loading Machine Learning Models

This notebook demonstrates different methods for saving and loading machine learning models, comparing various formats and best practices.

**Topics covered:**
- Using pickle for model serialization
- Using joblib for more efficient serialization
- scikit-learn's built-in persistence methods
- Saving/loading TensorFlow/Keras models
- Working with the ONNX format
- Saving model metadata and versioning best practices

Let's get started by setting up our environment and training a simple model.

## Import Required Libraries

Let's import all the necessary libraries for model training, saving, and loading.

In [ ]:
# Standard libraries
import os
import pickle
import time
import datetime
import json

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.datasets import load_iris, load_diabetes, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report
from sklearn.pipeline import Pipeline

# Model persistence
import joblib

# For ONNX conversion
try:
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType
    import onnxruntime as rt
    ONNX_AVAILABLE = True
except ImportError:
    ONNX_AVAILABLE = False
    print("ONNX libraries not available. ONNX examples will be skipped.")
    
# For TensorFlow/Keras examples
try:
    import tensorflow as tf
    from tensorflow import keras
    TENSORFLOW_AVAILABLE = True
except ImportError:
    TENSORFLOW_AVAILABLE = False
    print("TensorFlow not available. TensorFlow/Keras examples will be skipped.")

## Train a Simple ML Model

First, let's create and train a simple machine learning model using the Iris dataset from scikit-learn. We'll use this model throughout the notebook to demonstrate different saving and loading techniques.

In [ ]:
# Load the Iris dataset
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create and train a Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Evaluate the model
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Random Forest model accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

# Create a directory to save our models if it doesn't exist
os.makedirs('models', exist_ok=True)

## Saving Models with pickle

Python's built-in `pickle` module provides a simple way to serialize Python objects, including machine learning models, and save them to disk. This is the most basic approach to saving models.

In [ ]:
# Save the model using pickle
start_time = time.time()
with open('models/rf_model.pkl', 'wb') as file:
    pickle.dump(rf_model, file)
pickle_save_time = time.time() - start_time

# Check the file size
pickle_size = os.path.getsize('models/rf_model.pkl') / 1024  # Size in KB
print(f"Model saved using pickle in {pickle_save_time:.4f} seconds")
print(f"File size: {pickle_size:.2f} KB")

## Loading Models with pickle

Now let's load the model we just saved with pickle and verify that it works correctly.

In [ ]:
# Load the model using pickle
start_time = time.time()
with open('models/rf_model.pkl', 'rb') as file:
    loaded_model = pickle.load(file)
pickle_load_time = time.time() - start_time

# Verify that the loaded model works correctly
loaded_pred = loaded_model.predict(X_test)
loaded_accuracy = accuracy_score(y_test, loaded_pred)

print(f"Model loaded using pickle in {pickle_load_time:.4f} seconds")
print(f"Original model accuracy: {accuracy:.4f}")
print(f"Loaded model accuracy: {loaded_accuracy:.4f}")

# Make sure the predictions are identical
print(f"Predictions identical to original model: {np.array_equal(y_pred, loaded_pred)}")

## Saving Models with joblib

The `joblib` library provides an alternative to pickle that is more efficient for objects containing large NumPy arrays, which is common in machine learning models. It's particularly useful for scikit-learn models.

In [ ]:
# Save the model using joblib
start_time = time.time()
joblib.dump(rf_model, 'models/rf_model.joblib')
joblib_save_time = time.time() - start_time

# Check the file size
joblib_size = os.path.getsize('models/rf_model.joblib') / 1024  # Size in KB
print(f"Model saved using joblib in {joblib_save_time:.4f} seconds")
print(f"File size: {joblib_size:.2f} KB")

# Compare with pickle
print("\nComparison with pickle:")
print(f"Time ratio (joblib/pickle): {joblib_save_time/pickle_save_time:.2f}")
print(f"Size ratio (joblib/pickle): {joblib_size/pickle_size:.2f}")

## Loading Models with joblib

Let's load the model we saved with joblib and compare its performance with pickle.

In [ ]:
# Load the model using joblib
start_time = time.time()
joblib_model = joblib.load('models/rf_model.joblib')
joblib_load_time = time.time() - start_time

# Verify that the loaded model works correctly
joblib_pred = joblib_model.predict(X_test)
joblib_accuracy = accuracy_score(y_test, joblib_pred)

print(f"Model loaded using joblib in {joblib_load_time:.4f} seconds")
print(f"Original model accuracy: {accuracy:.4f}")
print(f"Loaded model accuracy: {joblib_accuracy:.4f}")

# Compare loading times with pickle
print(f"\nLoading time ratio (joblib/pickle): {joblib_load_time/pickle_load_time:.2f}")

# Make sure the predictions are identical
print(f"Predictions identical to original model: {np.array_equal(y_pred, joblib_pred)}")

## Model Formats in scikit-learn

Scikit-learn integrates with both pickle and joblib for model persistence. Let's explore some of the built-in model persistence functionality in scikit-learn, including how to save pipelines that include preprocessing steps.

In [ ]:
# Create a pipeline with preprocessing and a model
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Train the pipeline
pipeline.fit(X_train, y_train)

# Evaluate the pipeline
pipe_pred = pipeline.predict(X_test)
pipe_accuracy = accuracy_score(y_test, pipe_pred)
print(f"Pipeline accuracy: {pipe_accuracy:.4f}")

# Save the pipeline with joblib
joblib.dump(pipeline, 'models/rf_pipeline.joblib')
print("Pipeline saved successfully!")

# Load the pipeline
loaded_pipeline = joblib.load('models/rf_pipeline.joblib')
loaded_pipe_pred = loaded_pipeline.predict(X_test)
loaded_pipe_accuracy = accuracy_score(y_test, loaded_pipe_pred)

print(f"Loaded pipeline accuracy: {loaded_pipe_accuracy:.4f}")
print(f"Predictions identical: {np.array_equal(pipe_pred, loaded_pipe_pred)}")

## Saving and Loading Models in TensorFlow/Keras

If you're working with neural networks using TensorFlow/Keras, there are several formats available for saving and loading models. The most common ones are:

1. HDF5 format (.h5) - The whole model in a single file
2. SavedModel format - A directory containing the complete TensorFlow program
3. TensorFlow Lite - For deployment on mobile and edge devices

Let's demonstrate these methods by creating and saving a simple neural network model.

In [ ]:
if TENSORFLOW_AVAILABLE:
    # Create a simple neural network for the Iris dataset
    # Convert to one-hot encoding for multi-class classification
    y_train_one_hot = tf.keras.utils.to_categorical(y_train)
    y_test_one_hot = tf.keras.utils.to_categorical(y_test)
    
    # Normalize the input data
    X_train_norm = StandardScaler().fit_transform(X_train)
    X_test_norm = StandardScaler().fit_transform(X_test)
    
    # Build a simple model
    model = keras.Sequential([
        keras.layers.Dense(16, activation='relu', input_shape=(X_train.shape[1],)),
        keras.layers.Dense(8, activation='relu'),
        keras.layers.Dense(3, activation='softmax')
    ])
    
    # Compile the model
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    
    # Train the model
    model.fit(X_train_norm, y_train_one_hot, epochs=50, batch_size=16, verbose=0)
    
    # Evaluate the model
    loss, accuracy = model.evaluate(X_test_norm, y_test_one_hot, verbose=0)
    print(f"Neural network accuracy: {accuracy:.4f}")
    
    # 1. Save model in HDF5 format
    model.save('models/keras_model.h5')
    print("Model saved in HDF5 format")
    
    # 2. Save model in SavedModel format
    model.save('models/keras_saved_model')
    print("Model saved in SavedModel format")
    
    # 3. Convert to TensorFlow Lite
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter.convert()
    
    # Save the TF Lite model
    with open('models/model.tflite', 'wb') as f:
        f.write(tflite_model)
    print("Model saved in TensorFlow Lite format")
    
    # Load HDF5 model
    loaded_h5_model = keras.models.load_model('models/keras_model.h5')
    loss_h5, accuracy_h5 = loaded_h5_model.evaluate(X_test_norm, y_test_one_hot, verbose=0)
    print(f"\nLoaded HDF5 model accuracy: {accuracy_h5:.4f}")
    
    # Load SavedModel
    loaded_saved_model = keras.models.load_model('models/keras_saved_model')
    loss_saved, accuracy_saved = loaded_saved_model.evaluate(X_test_norm, y_test_one_hot, verbose=0)
    print(f"Loaded SavedModel accuracy: {accuracy_saved:.4f}")
else:
    print("TensorFlow is not available. Skipping Keras model saving examples.")

## Working with ONNX Format

The Open Neural Network Exchange (ONNX) is an open standard for representing machine learning models. It enables models to be transferred between different frameworks like PyTorch, TensorFlow, scikit-learn, and many others.

Let's convert our scikit-learn RandomForest model to the ONNX format and then use it for inference.

In [ ]:
if ONNX_AVAILABLE:
    # Convert the scikit-learn model to ONNX
    initial_type = [('float_input', FloatTensorType([None, X_train.shape[1]]))]
    onx = convert_sklearn(rf_model, initial_types=initial_type)
    
    # Save the ONNX model
    with open("models/rf_model.onnx", "wb") as f:
        f.write(onx.SerializeToString())
    
    print("Model successfully converted to ONNX and saved")
    
    # Create an ONNX inference session
    sess = rt.InferenceSession("models/rf_model.onnx")
    input_name = sess.get_inputs()[0].name
    label_name = sess.get_outputs()[0].name
    
    # Run inference
    pred_onx = sess.run([label_name], {input_name: X_test.astype(np.float32)})[0]
    
    # Check if predictions match
    onnx_accuracy = accuracy_score(y_test, pred_onx)
    print(f"ONNX model accuracy: {onnx_accuracy:.4f}")
    print(f"Predictions identical to original model: {np.array_equal(y_pred, pred_onx)}")
else:
    print("ONNX libraries are not available. Skipping ONNX examples.")

## Saving Model Metadata

When saving models for production use, it's important to also save metadata about the model such as:

1. Feature names and their descriptions
2. Training dataset information
3. Model hyperparameters
4. Preprocessing steps
5. Performance metrics
6. Model version and creation date

Let's demonstrate how to save this metadata alongside our model.

In [ ]:
# Create a dictionary with model metadata
model_metadata = {
    'model_type': 'RandomForestClassifier',
    'creation_date': datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    'sklearn_version': pd.__version__,
    'feature_names': list(feature_names),
    'target_names': list(target_names),
    'hyperparameters': rf_model.get_params(),
    'performance': {
        'accuracy': float(accuracy),
        'test_size': 0.2,
        'random_state': 42
    },
    'dataset': {
        'name': 'Iris',
        'n_samples': X.shape[0],
        'n_features': X.shape[1]
    }
}

# Save metadata as JSON
with open('models/rf_model_metadata.json', 'w') as f:
    json.dump(model_metadata, f, indent=2)

print("Model metadata saved successfully!")

# Load and display the metadata
with open('models/rf_model_metadata.json', 'r') as f:
    loaded_metadata = json.load(f)

print("\nModel Metadata:")
for key, value in loaded_metadata.items():
    if key != 'hyperparameters':  # Skip the verbose hyperparameters dictionary
        print(f"{key}: {value}")
print("Hyperparameters: {n_estimators: 100, random_state: 42, ...}")  # Simplified output

## Best Practices for Model Versioning

In a production environment, it's crucial to track different versions of your models and maintain a proper versioning system. Here are some best practices for model versioning:

### Model Versioning Best Practices

1. **Use a standardized naming convention**:
   - Include model type, version number, and date
   - Example: `rf_classifier_v1.2_2023-05-15.joblib`

2. **Create a directory structure that supports versioning**:
   ```
   models/
   ├── production/
   │   └── current_model.joblib
   ├── v1.0/
   │   ├── model.joblib
   │   └── metadata.json
   ├── v1.1/
   │   ├── model.joblib
   │   └── metadata.json
   └── archive/
       └── old_models/
   ```

3. **Store model artifacts consistently**:
   - Save the model file
   - Save the preprocessing pipeline
   - Save metadata (JSON or YAML)
   - Save sample predictions
   - Save evaluation metrics

4. **Use a model registry or tracking system**:
   - MLflow
   - DVC (Data Version Control)
   - Weights & Biases
   - TensorFlow Model Registry

5. **Include information about the training data**:
   - Dataset version or timestamp
   - Data processing steps
   - Train/test split information

6. **Document model lineage**:
   - Models that were used as starting points
   - Transfer learning sources
   - References to previous versions

7. **Implement a model approval workflow**:
   - Track which models have been approved for production
   - Store information about who approved them and when

In [ ]:
# Let's implement a simple versioning example

def save_versioned_model(model, version, metrics, dataset_info, description):
    """Save a model with version information"""
    # Create version directory if it doesn't exist
    version_dir = f"models/v{version}"
    os.makedirs(version_dir, exist_ok=True)
    
    # Save the model
    model_path = f"{version_dir}/model.joblib"
    joblib.dump(model, model_path)
    
    # Save metadata
    metadata = {
        'version': version,
        'saved_at': datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'description': description,
        'metrics': metrics,
        'dataset_info': dataset_info,
        'feature_names': list(feature_names),
        'model_params': model.get_params()
    }
    
    with open(f"{version_dir}/metadata.json", 'w') as f:
        json.dump(metadata, f, indent=2)
    
    print(f"Model version {version} saved successfully!")
    return version_dir

# Save our current Random Forest model as version 1.0
metrics = {
    'accuracy': float(accuracy),
    'classification_report': classification_report(y_test, y_pred, output_dict=True)
}

dataset_info = {
    'name': 'Iris',
    'shape': X.shape,
    'train_test_split': 0.2,
    'random_state': 42
}

description = "Initial Random Forest model on Iris dataset"

version_dir = save_versioned_model(
    model=rf_model, 
    version="1.0", 
    metrics=metrics, 
    dataset_info=dataset_info,
    description=description
)

# Let's now create a "production" link to our latest model version
os.makedirs("models/production", exist_ok=True)
prod_model_path = "models/production/current_model.joblib"
prod_metadata_path = "models/production/current_metadata.json"

# Use symbolic links if on Unix or just copy on Windows
try:
    # Try to create symbolic links (works on Unix/Linux/macOS)
    if os.path.exists(prod_model_path):
        os.remove(prod_model_path)
    if os.path.exists(prod_metadata_path):
        os.remove(prod_metadata_path)
    
    os.symlink(
        os.path.abspath(f"{version_dir}/model.joblib"), 
        os.path.abspath(prod_model_path)
    )
    os.symlink(
        os.path.abspath(f"{version_dir}/metadata.json"), 
        os.path.abspath(prod_metadata_path)
    )
    print("Created symbolic links to production model")
except:
    # Fallback to copying files on Windows
    import shutil
    shutil.copy2(f"{version_dir}/model.joblib", prod_model_path)
    shutil.copy2(f"{version_dir}/metadata.json", prod_metadata_path)
    print("Copied files to production directory")

# Display the created file structure
def print_directory_tree(directory, prefix=""):
    contents = os.listdir(directory)
    files = [item for item in contents if os.path.isfile(os.path.join(directory, item))]
    dirs = [item for item in contents if os.path.isdir(os.path.join(directory, item))]
    
    # Print files
    for file in files:
        print(f"{prefix}├── {file}")
    
    # Print directories
    for i, dir_name in enumerate(dirs):
        is_last_dir = i == len(dirs) - 1
        print(f"{prefix}{'└── ' if is_last_dir and not files else '├── '}{dir_name}/")
        
        # Only go one level deeper
        if directory.count(os.sep) < 2:  # Limit depth
            next_prefix = prefix + ('    ' if is_last_dir else '│   ')
            sub_path = os.path.join(directory, dir_name)
            sub_contents = os.listdir(sub_path)
            
            # Only show a few items if there are many
            if len(sub_contents) > 5:
                for sub_item in sub_contents[:3]:
                    print(f"{next_prefix}├── {sub_item}")
                print(f"{next_prefix}└── ... ({len(sub_contents)-3} more items)")
            else:
                for j, sub_item in enumerate(sub_contents):
                    is_last = j == len(sub_contents) - 1
                    print(f"{next_prefix}{'└── ' if is_last else '├── '}{sub_item}")

print("\nModel Directory Structure:")
print("models/")
print_directory_tree("models")

## Summary

In this notebook, we've covered various methods for saving and loading machine learning models:

1. **Basic Serialization**:
   - Using Python's built-in `pickle` module
   - Using `joblib` for more efficient serialization of scikit-learn models

2. **Framework-specific Formats**:
   - scikit-learn Pipeline serialization
   - TensorFlow/Keras formats (HDF5, SavedModel, TF Lite)

3. **Interoperable Formats**:
   - ONNX for cross-framework compatibility

4. **Best Practices**:
   - Saving model metadata
   - Implementing versioning strategies
   - Directory structure for model management

Key takeaways:
- `joblib` is generally preferred for scikit-learn models over `pickle`
- Always save preprocessing steps alongside the model (using pipelines)
- Save comprehensive metadata with your models
- Implement a proper versioning system for production models
- Consider framework-specific optimized formats or interoperable formats like ONNX for deployment